# FlowSure AI Support Assistant

**Project 4: AI-powered ticketsysteem voor klantenondersteuning**  
Vak: MLOps  
Beroepsproduct: end-to-end MLOps-pipeline voor ticketclassificatie, antwoordgeneratie, deployment en monitoring.

## Samenvatting
FlowSure is een snelgroeiend SaaS-bedrijf dat veel supporttickets ontvangt via e-mail, chat, webformulieren en in-app support. In dit notebook bouwen we een prototype van een AI-assisted ticketsysteem met twee modellen:

1. **Edge model:** een lichtgewicht classificatiemodel dat tickets indeelt op categorie/intent en prioriteit.
2. **Cloud model:** een generatieve/RAG-achtige component die een conceptantwoord ophaalt of samenstelt op basis van historische supportdata.

Het notebook bevat:
- product- en stakeholderanalyse;
- datavereisten en modelvereisten;
- data ingestion en preprocessing;
- baseline en verbeterd classificatiemodel;
- experiment tracking;
- deploymentvoorbeeld met FastAPI;
- monitoring op data, modelkwaliteit, latency en drift;
- governance, privacy en GAI-verantwoording.

## 0. Installatie en imports

De eerste codecel installeert alleen ontbrekende libraries. In een schoolomgeving met Anaconda zijn `pandas`, `numpy`, `scikit-learn`, `matplotlib` en `joblib` meestal al aanwezig.

> Let op: dit notebook is zo opgezet dat het lokaal kan draaien. De grote Twitter-dataset (`twcs.csv`) wordt standaard gesampled om runtime en geheugen te beperken.

In [ ]:
# Prompt 1: Compleet MLOps notebook voor AI-powered ticketsysteem FlowSure
# Deze cel installeert ontbrekende libraries indien nodig.

import importlib
import subprocess
import sys

REQUIRED_PACKAGES = [
    "pandas",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "joblib",
]


def install_if_missing(package_name: str) -> None:
    """Installeer een Python-package als deze nog niet beschikbaar is."""
    try:
        importlib.import_module(package_name.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


for package in REQUIRED_PACKAGES:
    install_if_missing(package)

In [ ]:
# Standaard imports

from __future__ import annotations

import json
import os
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

## 1. Productoverzicht

### 1.1 Probleem
Supportmedewerkers van FlowSure moeten tickets nu grotendeels handmatig lezen, categoriseren, prioriteren, routeren en beantwoorden. Dit zorgt voor:

- lange wachttijden tijdens piekmomenten;
- inconsistente ticketclassificatie;
- hogere operationele kosten;
- minder tijd voor complexe klantcases;
- minder snelle signalering van bugs en productfrictie.

### 1.2 Doel
Het doel is een AI-assisted supportoplossing die tickets automatisch classificeert en agents een conceptantwoord geeft. De agent blijft eindverantwoordelijk en controleert het AI-antwoord voordat het naar de klant gaat.

### 1.3 Scope
Binnen dit prototype bouwen we:

- een data ingestion pipeline voor Bitext- en Twitter-supportdata;
- een edge classificatiemodel voor categorie/intent;
- een eenvoudige prioriteitsinschatting;
- een RAG-achtig conceptantwoord op basis van vergelijkbare historische tickets;
- monitoring voor datakwaliteit, drift, latency en modelprestatie;
- deploymentvoorbeeld met FastAPI en Docker.

### 1.4 Buiten scope
Niet volledig uitgewerkt in dit prototype:

- echte koppeling met e-mail, chat of CRM-systemen;
- productieklare cloudinfrastructuur;
- volledige juridische DPIA;
- menselijke evaluatie door echte supportagents.

## 2. Stakeholderanalyse

| Stakeholder | Belang | Verwachting van het systeem |
|---|---|---|
| Supportagents | Sneller tickets verwerken | Betrouwbare suggesties die aanpasbaar zijn |
| Teamleads | SLA's en escalaties bewaken | Inzicht in urgentie, volume en doorlooptijd |
| Klanten | Snelle en juiste antwoorden | Kortere wachttijd en consistente kwaliteit |
| IT/Security | Veilige integratie | Logging, toegangsbeheer en veilige deployment |
| Legal/Compliance | AVG/GDPR-naleving | Dataminimalisatie, retentiebeleid en anonimisatie |
| Productteams | Bugs en UX-frictie herkennen | Trendrapportages per productmodule |
| Management | Kosten en klanttevredenheid | Efficiëntere supportorganisatie en dashboards |

## 3. Vereisten

### 3.1 Datavereisten

| Eis | Specificatie |
|---|---|
| Kwaliteit | Geen lege berichten, duplicaten verwijderen, consistente kolomnamen |
| Volume | Prototype werkt met samples; ontwerp is schaalbaar naar miljoenen tickets |
| Snelheid | Nieuwe tickets moeten binnen seconden verwerkt kunnen worden |
| Privacy | Namen, e-mails, telefoonnummers, mentions en URLs anonimiseren |
| Veiligheid | Geen ruwe persoonsgegevens in logs of modelartefacten |
| Retentie | Alleen noodzakelijke data bewaren |
| Monitoring | Drift, ontbrekende waarden en gewijzigde ticketverdeling detecteren |

### 3.2 Modelvereisten

| Model | Taak | Belangrijkste eisen |
|---|---|---|
| Edge model | Ticketclassificatie | Lichtgewicht, snel, uitlegbaar, deploybaar via API |
| Cloud model | Conceptantwoord | Gebaseerd op historische antwoorden of kennisbank, agent review verplicht |

### 3.3 Operationele vereisten

- CI/CD via GitHub Actions of vergelijkbare tool.
- Modelversies opslaan met model registry of bestandssysteem.
- Rollback mogelijk naar vorige modelversie.
- Monitoringdashboard voor support- en modelmetrics.
- Hertraining bij datadrift of lagere performance.

## 4. Configuratie

Hier definiëren we vaste paden en instellingen. De grote Twitter-dataset wordt standaard beperkt met `MAX_TWCS_ROWS`, omdat het bestand erg groot kan zijn.

In [ ]:
@dataclass(frozen=True)
class ProjectConfig:
    """Centrale configuratie voor het FlowSure-project.

    Gebruik bij lokaal draaien dezelfde map als dit notebook, of maak een mapje
    `data/` en zet de drie CSV-bestanden daarin.
    """

    bitext_filename: str = "Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
    twitter_sample_filename: str = "sample.csv"
    twitter_full_filename: str = "twcs.csv"
    output_dir: Path = Path("./flowsure_artifacts")
    random_state: int = 42
    test_size: float = 0.20
    validation_size: float = 0.20
    max_twcs_rows: int = 50_000
    min_text_length: int = 5

    def find_file(self, filename: str) -> Path:
        """Zoek een databestand op meerdere logische locaties.

        Dit voorkomt fouten wanneer het notebook lokaal draait in VS Code.
        In ChatGPT stond de data in /mnt/data, maar op je eigen laptop bestaat
        dat pad meestal niet. Daarom zoekt deze functie ook in de huidige map
        en in ./data.
        """
        search_dirs = [
            Path.cwd(),
            Path.cwd() / "data",
            Path.cwd().parent,
            Path("/mnt/data"),
        ]

        for directory in search_dirs:
            candidate = directory / filename
            if candidate.exists():
                return candidate

        raise FileNotFoundError(
            "Bestand niet gevonden: "
            f"{filename}

"
            "Oplossing:
"
            "1. Zet de drie CSV-bestanden in dezelfde map als dit notebook, of
"
            "2. maak een map `data/` naast dit notebook en zet de bestanden daarin.

"
            "Benodigde bestanden:
"
            f"- {self.bitext_filename}
"
            f"- {self.twitter_sample_filename}
"
            f"- {self.twitter_full_filename}"
        )

    @property
    def bitext_path(self) -> Path:
        return self.find_file(self.bitext_filename)

    @property
    def twitter_sample_path(self) -> Path:
        return self.find_file(self.twitter_sample_filename)

    @property
    def twitter_full_path(self) -> Path:
        return self.find_file(self.twitter_full_filename)


config = ProjectConfig()
config.output_dir.mkdir(parents=True, exist_ok=True)

print("Outputmap:", config.output_dir.resolve())
print("Bitext-pad:", config.bitext_path)
print("Twitter sample-pad:", config.twitter_sample_path)
print("Twitter full-pad:", config.twitter_full_path)


## 5. Data ingestion

We laden drie bestanden:

1. `Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv`  
   Bevat klantvragen, categorieën, intenties en voorbeeldantwoorden.

2. `sample.csv`  
   Kleine sample van Twitter Customer Support-data.

3. `twcs.csv`  
   Grote Twitter Customer Support-dataset. Deze wordt beperkt ingelezen.

Alle bronnen worden genormaliseerd naar één ticketstructuur.

In [ ]:
class DataIngestion:
    """Laadt en normaliseert FlowSure supportdata uit meerdere bronnen."""

    def __init__(self, cfg: ProjectConfig) -> None:
        self.cfg = cfg

    @staticmethod
    def validate_file(path: Path) -> None:
        """Controleer of een bestand bestaat."""
        if not path.exists():
            raise FileNotFoundError(f"Bestand niet gevonden: {path}")

    def load_bitext(self) -> pd.DataFrame:
        """Laad Bitext customer support dataset."""
        self.validate_file(self.cfg.bitext_path)
        df = pd.read_csv(self.cfg.bitext_path)
        df["source"] = "bitext"
        return df

    def load_twitter_sample(self) -> pd.DataFrame:
        """Laad kleine Twitter sample."""
        self.validate_file(self.cfg.twitter_sample_path)
        df = pd.read_csv(self.cfg.twitter_sample_path)
        df["source"] = "twitter_sample"
        return df

    def load_twitter_full(self) -> pd.DataFrame:
        """Laad maximaal N rijen van de grote Twitter supportdataset."""
        self.validate_file(self.cfg.twitter_full_path)
        df = pd.read_csv(self.cfg.twitter_full_path, nrows=self.cfg.max_twcs_rows)
        df["source"] = "twitter_full"
        return df


loader = DataIngestion(config)
bitext_raw = loader.load_bitext()
twitter_sample_raw = loader.load_twitter_sample()
twitter_raw = loader.load_twitter_full()

print("Bitext:", bitext_raw.shape)
print("Twitter sample:", twitter_sample_raw.shape)
print("Twitter full sample:", twitter_raw.shape)

In [ ]:
# Inspectie van kolommen en eerste rijen

display(bitext_raw.head())
display(twitter_sample_raw.head())
display(twitter_raw.head())

## 6. Data normalisatie en privacy preprocessing

Voor een MLOps-pipeline is een eenduidig schema belangrijk. We zetten alle bronnen om naar:

| Kolom | Betekenis |
|---|---|
| ticket_id | Unieke identificatie |
| source | Databron |
| channel | Kanaal, bijvoorbeeld chatbot of Twitter |
| created_at | Datum/tijd indien beschikbaar |
| customer_message | Bericht van klant |
| agent_response | Antwoord van agent of voorbeeldantwoord |
| category | Categorie of onderwerp |
| intent | Intentie |
| priority | Ingeschatte prioriteit |
| product_module | Ingeschatte productmodule |

Daarnaast anonimiseren we persoonsgegevens en platforminformatie.

In [ ]:
class TextPreprocessor:
    """Tekstopschoning en privacy-anonimisatie."""

    EMAIL_PATTERN = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
    URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
    PHONE_PATTERN = re.compile(r"\b(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{2,4}\)?[-.\s]?)?\d{3,4}[-.\s]?\d{3,4}\b")
    MENTION_PATTERN = re.compile(r"@\w+")
    ORDER_PATTERN = re.compile(r"\{\{[^}]+\}\}")

    @classmethod
    def anonymize(cls, text: object) -> str:
        """Anonimiseer gevoelige patronen in tekst."""
        if pd.isna(text):
            return ""
        clean_text = str(text)
        clean_text = cls.EMAIL_PATTERN.sub("[EMAIL]", clean_text)
        clean_text = cls.URL_PATTERN.sub("[URL]", clean_text)
        clean_text = cls.PHONE_PATTERN.sub("[PHONE]", clean_text)
        clean_text = cls.MENTION_PATTERN.sub("[MENTION]", clean_text)
        clean_text = cls.ORDER_PATTERN.sub("[PLACEHOLDER]", clean_text)
        return clean_text

    @staticmethod
    def normalize_text(text: object) -> str:
        """Normaliseer tekst voor ML-training."""
        if pd.isna(text):
            return ""
        clean_text = str(text).lower().strip()
        clean_text = re.sub(r"\s+", " ", clean_text)
        return clean_text

    @classmethod
    def clean(cls, text: object) -> str:
        """Combineer anonimisatie en normalisatie."""
        return cls.normalize_text(cls.anonymize(text))

In [ ]:
def assign_priority(text: str, category: str = "") -> str:
    """Bepaal een simpele priority-label op basis van trefwoorden.

    In productie zou dit een apart model of business-rule engine zijn.
    """
    text_combined = f"{text} {category}".lower()
    high_keywords = [
        "urgent", "asap", "immediately", "down", "downtime", "outage",
        "not working", "can't access", "cannot access", "security", "breach",
        "payment failed", "refund", "cancel", "angry", "complaint"
    ]
    medium_keywords = [
        "issue", "problem", "error", "bug", "failed", "invoice", "billing"
    ]

    if any(keyword in text_combined for keyword in high_keywords):
        return "high"
    if any(keyword in text_combined for keyword in medium_keywords):
        return "medium"
    return "low"


def assign_product_module(text: str, category: str = "") -> str:
    """Schat productmodule op basis van eenvoudige keywordregels."""
    text_combined = f"{text} {category}".lower()
    module_keywords = {
        "billing": ["billing", "invoice", "payment", "refund", "charge", "subscription"],
        "account": ["account", "login", "password", "access", "profile"],
        "orders": ["order", "cancel", "delivery", "shipping"],
        "integrations": ["api", "integration", "sync", "webhook"],
        "planning": ["schedule", "calendar", "planning", "booking"],
    }
    for module, keywords in module_keywords.items():
        if any(keyword in text_combined for keyword in keywords):
            return module
    return "general"


def normalize_bitext(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliseer Bitext-data naar het ticketschema."""
    normalized = pd.DataFrame({
        "ticket_id": [f"bitext_{idx}" for idx in df.index],
        "source": df.get("source", "bitext"),
        "channel": "chatbot_training",
        "created_at": pd.NaT,
        "customer_message": df["instruction"].apply(TextPreprocessor.clean),
        "agent_response": df["response"].apply(TextPreprocessor.clean),
        "category": df["category"].astype(str).str.lower().str.strip(),
        "intent": df["intent"].astype(str).str.lower().str.strip(),
    })
    normalized["priority"] = normalized.apply(
        lambda row: assign_priority(row["customer_message"], row["category"]), axis=1
    )
    normalized["product_module"] = normalized.apply(
        lambda row: assign_product_module(row["customer_message"], row["category"]), axis=1
    )
    return normalized


def normalize_twitter(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """Normaliseer Twitter-data naar het ticketschema.

    Alleen inbound tweets worden als klanttickets gebruikt. Omdat deze dataset
    geen handmatige categorieën bevat, labelen we deze als 'social_support'.
    """
    inbound = df[df["inbound"] == True].copy()
    normalized = pd.DataFrame({
        "ticket_id": inbound["tweet_id"].apply(lambda value: f"{source_name}_{value}"),
        "source": source_name,
        "channel": "twitter",
        "created_at": pd.to_datetime(inbound["created_at"], errors="coerce"),
        "customer_message": inbound["text"].apply(TextPreprocessor.clean),
        "agent_response": "",
        "category": "social_support",
        "intent": "unknown",
    })
    normalized["priority"] = normalized.apply(
        lambda row: assign_priority(row["customer_message"], row["category"]), axis=1
    )
    normalized["product_module"] = normalized.apply(
        lambda row: assign_product_module(row["customer_message"], row["category"]), axis=1
    )
    return normalized


bitext_tickets = normalize_bitext(bitext_raw)
twitter_sample_tickets = normalize_twitter(twitter_sample_raw, "twitter_sample")
twitter_tickets = normalize_twitter(twitter_raw, "twitter_full")

all_tickets = pd.concat(
    [bitext_tickets, twitter_sample_tickets, twitter_tickets],
    ignore_index=True,
)

all_tickets = all_tickets.drop_duplicates(subset=["customer_message"])
all_tickets = all_tickets[
    all_tickets["customer_message"].str.len() >= config.min_text_length
].reset_index(drop=True)

print(all_tickets.shape)
display(all_tickets.head())

## 7. Datakwaliteit en EDA

We controleren:

- missende waarden;
- verdeling van bronnen;
- verdeling van categorieën;
- prioriteiten;
- tekstlengte.

In [ ]:
def data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    """Maak een compact datakwaliteitsrapport."""
    report = pd.DataFrame({
        "column": df.columns,
        "missing_values": df.isna().sum().values,
        "missing_percentage": (df.isna().mean().values * 100).round(2),
        "unique_values": df.nunique(dropna=True).values,
        "dtype": [str(dtype) for dtype in df.dtypes],
    })
    return report


quality_report = data_quality_report(all_tickets)
display(quality_report)

In [ ]:
# Verdelingen

display(all_tickets["source"].value_counts())
display(all_tickets["category"].value_counts().head(20))
display(all_tickets["priority"].value_counts())
display(all_tickets["product_module"].value_counts())

In [ ]:
# Visualisaties

all_tickets["text_length"] = all_tickets["customer_message"].str.len()

plt.figure(figsize=(10, 5))
all_tickets["source"].value_counts().plot(kind="bar")
plt.title("Aantal tickets per bron")
plt.xlabel("Bron")
plt.ylabel("Aantal tickets")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
all_tickets["priority"].value_counts().plot(kind="bar")
plt.title("Verdeling van prioriteiten")
plt.xlabel("Prioriteit")
plt.ylabel("Aantal tickets")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
all_tickets["text_length"].hist(bins=50)
plt.title("Verdeling van ticketlengte")
plt.xlabel("Aantal karakters")
plt.ylabel("Aantal tickets")
plt.tight_layout()
plt.show()

## 8. Train/validatie/test-split

Voor het edge model gebruiken we vooral de Bitext-data, omdat deze echte labels bevat voor categorie en intent. De Twitter-data is nuttig voor realistische klanttaal en monitoring, maar heeft geen betrouwbare categorie/intent-labels.

We trainen daarom primair op Bitext-categorieën. Twitter-data kan later worden gebruikt voor semi-supervised learning of handmatige labeling.

In [ ]:
model_data = all_tickets[
    (all_tickets["source"] == "bitext") &
    (all_tickets["category"].notna()) &
    (all_tickets["customer_message"].str.len() >= config.min_text_length)
].copy()

X = model_data["customer_message"]
y = model_data["category"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=config.test_size + config.validation_size,
    random_state=config.random_state,
    stratify=y,
)

relative_test_size = config.test_size / (config.test_size + config.validation_size)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=relative_test_size,
    random_state=config.random_state,
    stratify=y_temp,
)

print("Train:", X_train.shape)
print("Validatie:", X_val.shape)
print("Test:", X_test.shape)
print("Aantal categorieën:", y.nunique())

## 9. Modellering: Edge model

We trainen twee lichtgewicht modellen:

1. **TF-IDF + Logistic Regression**  
   Uitlegbaar, snel en geschikt als baseline.

2. **TF-IDF + Linear SVM**  
   Vaak sterk voor tekstclassificatie, ook relatief lichtgewicht.

De beste wordt geselecteerd op macro F1-score. Macro F1 is belangrijk omdat supportcategorieën ongelijk verdeeld kunnen zijn.

In [ ]:
class EdgeTicketClassifier:
    """Wrapper voor een lichtgewicht ticketclassificatiemodel."""

    def __init__(self, model_name: str, pipeline: Pipeline) -> None:
        self.model_name = model_name
        self.pipeline = pipeline
        self.metrics: Dict[str, float] = {}

    def train(self, x_train: pd.Series, y_train: pd.Series) -> None:
        """Train het model."""
        self.pipeline.fit(x_train, y_train)

    def predict(self, x_values: pd.Series) -> np.ndarray:
        """Voorspel categorieën."""
        return self.pipeline.predict(x_values)

    def evaluate(self, x_values: pd.Series, y_true: pd.Series) -> Dict[str, float]:
        """Evalueer model met accuracy en macro F1."""
        y_pred = self.predict(x_values)
        self.metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        }
        return self.metrics

    def save(self, path: Path) -> None:
        """Sla het modelartefact op."""
        joblib.dump(self.pipeline, path)


models = [
    EdgeTicketClassifier(
        "tfidf_logistic_regression",
        Pipeline([
            ("tfidf", TfidfVectorizer(max_features=20_000, ngram_range=(1, 2))),
            ("clf", LogisticRegression(max_iter=1_000, class_weight="balanced")),
        ]),
    ),
    EdgeTicketClassifier(
        "tfidf_linear_svc",
        Pipeline([
            ("tfidf", TfidfVectorizer(max_features=20_000, ngram_range=(1, 2))),
            ("clf", LinearSVC(class_weight="balanced")),
        ]),
    ),
]

experiment_results = []

for model in models:
    start_time = time.perf_counter()
    model.train(X_train, y_train)
    training_time = time.perf_counter() - start_time
    val_metrics = model.evaluate(X_val, y_val)

    result = {
        "model_name": model.model_name,
        "training_time_seconds": round(training_time, 3),
        **{key: round(value, 4) for key, value in val_metrics.items()},
    }
    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results).sort_values("macro_f1", ascending=False)
display(results_df)

In [ ]:
# Selecteer beste model op macro F1-score

best_model_name = results_df.iloc[0]["model_name"]
best_model = next(model for model in models if model.model_name == best_model_name)

print("Beste model:", best_model.model_name)
print("Validatiemetrics:", best_model.metrics)

## 10. Testevaluatie

Nu evalueren we het beste model op de testset, die nog niet is gebruikt voor modelselectie.

In [ ]:
test_metrics = best_model.evaluate(X_test, y_test)
y_test_pred = best_model.predict(X_test)

print("Testmetrics:")
print(json.dumps({key: round(value, 4) for key, value in test_metrics.items()}, indent=2))

print("\nClassification report:")
print(classification_report(y_test, y_test_pred))

In [ ]:
# Confusion matrix voor de meest voorkomende categorieën

top_labels = y_test.value_counts().head(10).index.tolist()
mask = y_test.isin(top_labels)

cm = confusion_matrix(y_test[mask], y_test_pred[mask], labels=top_labels)

plt.figure(figsize=(12, 8))
display_matrix = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=top_labels)
display_matrix.plot(xticks_rotation=45)
plt.title("Confusion matrix top 10 categorieën")
plt.tight_layout()
plt.show()

## 11. Latencytest

Een edge model moet snel genoeg zijn voor real-time tickettriage. We meten de gemiddelde inferentietijd per ticket.

In [ ]:
def measure_latency(model: EdgeTicketClassifier, samples: pd.Series, n_runs: int = 5) -> Dict[str, float]:
    """Meet gemiddelde inference latency."""
    latencies = []
    for _ in range(n_runs):
        start_time = time.perf_counter()
        _ = model.predict(samples)
        elapsed = time.perf_counter() - start_time
        latencies.append(elapsed / len(samples))

    return {
        "mean_latency_ms": float(np.mean(latencies) * 1000),
        "p95_latency_ms": float(np.percentile(latencies, 95) * 1000),
    }


latency_metrics = measure_latency(best_model, X_test.sample(min(500, len(X_test)), random_state=config.random_state))
print(json.dumps(latency_metrics, indent=2))

## 12. Prioriteit en routering

Naast categorie voorspellen we prioriteit en routering met business rules. In productie kan dit vervangen worden door aparte modellen.

In [ ]:
def route_ticket(category: str, product_module: str, priority: str) -> str:
    """Routeer ticket naar support squad."""
    if priority == "high":
        return "escalation_squad"
    if product_module == "billing":
        return "billing_squad"
    if product_module == "account":
        return "identity_access_squad"
    if product_module == "integrations":
        return "integration_squad"
    if product_module == "planning":
        return "planning_product_squad"
    if category in {"refund", "cancel_order", "order"}:
        return "order_support_squad"
    return "general_support_squad"


def classify_new_ticket(text: str) -> Dict[str, str]:
    """Volledige edge inference voor één ticket."""
    clean_text = TextPreprocessor.clean(text)
    predicted_category = best_model.predict(pd.Series([clean_text]))[0]
    priority = assign_priority(clean_text, predicted_category)
    product_module = assign_product_module(clean_text, predicted_category)
    squad = route_ticket(predicted_category, product_module, priority)

    return {
        "clean_text": clean_text,
        "predicted_category": predicted_category,
        "priority": priority,
        "product_module": product_module,
        "assigned_squad": squad,
    }


example_ticket = "Hi, I cannot access my account and our planning dashboard is down. This is urgent."
classify_new_ticket(example_ticket)

## 13. Cloud model: eenvoudige RAG/conceptantwoordcomponent

Voor het cloudmodel maken we een eenvoudige retrieval-aanpak:

1. Zoek historische tickets die lijken op het nieuwe ticket.
2. Haal het bijbehorende antwoord op.
3. Maak een conceptantwoord dat de agent kan controleren.

In productie kan deze stap worden vervangen door een cloud-LLM met RAG, waarbij kennisbankartikelen worden opgehaald uit bijvoorbeeld een vector database.

In [ ]:
class SimpleResponseRetriever:
    """Eenvoudige retrieval-component voor conceptantwoorden."""

    def __init__(self, knowledge_df: pd.DataFrame) -> None:
        self.knowledge_df = knowledge_df.dropna(subset=["customer_message", "agent_response"]).copy()
        self.knowledge_df = self.knowledge_df[self.knowledge_df["agent_response"].str.len() > 0]
        self.vectorizer = TfidfVectorizer(max_features=30_000, ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform(self.knowledge_df["customer_message"])

    def retrieve(self, query: str, top_k: int = 3) -> pd.DataFrame:
        """Zoek meest vergelijkbare historische tickets."""
        clean_query = TextPreprocessor.clean(query)
        query_vector = self.vectorizer.transform([clean_query])
        scores = (self.matrix @ query_vector.T).toarray().ravel()
        top_indices = scores.argsort()[::-1][:top_k]

        results = self.knowledge_df.iloc[top_indices].copy()
        results["similarity_score"] = scores[top_indices]
        return results[["customer_message", "agent_response", "category", "intent", "similarity_score"]]

    def generate_draft_response(self, query: str) -> str:
        """Maak een conceptantwoord op basis van het meest vergelijkbare ticket."""
        retrieved = self.retrieve(query, top_k=1)
        if retrieved.empty or retrieved.iloc[0]["similarity_score"] <= 0:
            return (
                "Bedankt voor uw bericht. We hebben uw vraag ontvangen en onderzoeken "
                "dit verder. Een supportmedewerker komt hier zo snel mogelijk op terug."
            )

        base_response = retrieved.iloc[0]["agent_response"]
        return (
            "Conceptantwoord voor agent review:\n\n"
            f"{base_response}\n\n"
            "Controleer dit antwoord op juistheid, klantcontext en beleid voordat het wordt verzonden."
        )


response_retriever = SimpleResponseRetriever(bitext_tickets)
retrieved_examples = response_retriever.retrieve(example_ticket, top_k=3)
display(retrieved_examples)
print(response_retriever.generate_draft_response(example_ticket))

## 14. Experiment tracking

In productie is MLflow een goede keuze voor experiment tracking. Omdat MLflow niet altijd standaard is geïnstalleerd, slaan we hier minimaal de experimentresultaten lokaal op als JSON en CSV.

Opgeslagen artefacten:

- modelmetrics;
- modelbestand;
- configuratie;
- datakwaliteitsrapport.

In [ ]:
# Artefacten opslaan

model_path = config.output_dir / "edge_ticket_classifier.joblib"
metrics_path = config.output_dir / "metrics.json"
results_path = config.output_dir / "experiment_results.csv"
quality_path = config.output_dir / "data_quality_report.csv"
config_path = config.output_dir / "config.json"

best_model.save(model_path)
results_df.to_csv(results_path, index=False)
quality_report.to_csv(quality_path, index=False)

metadata = {
    "best_model_name": best_model.model_name,
    "test_metrics": test_metrics,
    "latency_metrics": latency_metrics,
    "model_path": str(model_path),
    "created_at": pd.Timestamp.utcnow().isoformat(),
}

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

with open(config_path, "w", encoding="utf-8") as file:
    json.dump({key: str(value) for key, value in config.__dict__.items()}, file, indent=2)

print("Artefacten opgeslagen in:", config.output_dir.resolve())
print(os.listdir(config.output_dir))

## 15. Deployment met FastAPI

Onderstaande cel schrijft een eenvoudige `app.py` waarmee het edge model als API kan worden gedeployed.

Voorbeeld endpoints:

- `GET /health`
- `POST /predict`

Voor productie hoort hier authenticatie, rate limiting, logging en inputvalidatie bij.

In [ ]:
app_py = r"""
from pathlib import Path
from typing import Dict

import joblib
from fastapi import FastAPI
from pydantic import BaseModel

MODEL_PATH = Path("flowsure_artifacts/edge_ticket_classifier.joblib")
model = joblib.load(MODEL_PATH)

app = FastAPI(title="FlowSure AI Support Assistant")


class TicketRequest(BaseModel):
    text: str


class TicketPrediction(BaseModel):
    category: str


@app.get("/health")
def health() -> Dict[str, str]:
    return {"status": "ok"}


@app.post("/predict", response_model=TicketPrediction)
def predict(ticket: TicketRequest) -> TicketPrediction:
    prediction = model.predict([ticket.text])[0]
    return TicketPrediction(category=prediction)
"""

app_path = config.output_dir / "app.py"
with open(app_path, "w", encoding="utf-8") as file:
    file.write(app_py.strip() + "\n")

print(f"FastAPI-app geschreven naar: {app_path}")

In [ ]:
dockerfile = r"""
FROM python:3.11-slim

WORKDIR /app

COPY flowsure_artifacts/ /app/flowsure_artifacts/

RUN pip install --no-cache-dir fastapi uvicorn scikit-learn joblib numpy scipy

EXPOSE 8000

CMD ["uvicorn", "flowsure_artifacts.app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

dockerfile_path = Path("./Dockerfile")
with open(dockerfile_path, "w", encoding="utf-8") as file:
    file.write(dockerfile.strip() + "\n")

print("Dockerfile geschreven naar:", dockerfile_path.resolve())

### 15.1 Voorbeeld CI/CD met GitHub Actions

Deze workflow draait tests, bouwt een Docker-image en kan daarna naar een registry pushen. In dit prototype staat de deploystap als voorbeeld beschreven.

In [ ]:
github_actions_yaml = r"""
name: flowsure-mlops-ci-cd

on:
  push:
    branches: ["main"]
  pull_request:
    branches: ["main"]

jobs:
  test-build:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pandas numpy scikit-learn matplotlib joblib fastapi uvicorn pytest

      - name: Run basic checks
        run: |
          python -m compileall .

      - name: Build Docker image
        run: |
          docker build -t flowsure-ai-support:latest .
"""

workflow_dir = Path(".github/workflows")
workflow_dir.mkdir(parents=True, exist_ok=True)
workflow_path = workflow_dir / "ci_cd.yml"
with open(workflow_path, "w", encoding="utf-8") as file:
    file.write(github_actions_yaml.strip() + "\n")

print("Workflow geschreven naar:", workflow_path)

## 16. Monitoring

Voor monitoring kijken we naar vier lagen:

1. **Systeemmonitoring**: latency, foutpercentages, uptime.
2. **Datamonitoring**: ontbrekende waarden, tekstlengte, categorieverdeling.
3. **Modelmonitoring**: confidence/proxy, accuracy wanneer labels later beschikbaar komen.
4. **Businessmonitoring**: SLA-risico, escalaties, volumes per kanaal.

Omdat `LinearSVC` geen standaard predict probability geeft, gebruiken we voor monitoring vooral distributies, latency en drift. Als het beste model Logistic Regression is, kunnen probabilities ook gebruikt worden.

In [ ]:
class DataDriftMonitor:
    """Eenvoudige driftmonitor voor categorische distributies."""

    @staticmethod
    def population_stability_index(
        expected: pd.Series,
        actual: pd.Series,
        epsilon: float = 1e-6,
    ) -> float:
        """Bereken PSI tussen twee categorische verdelingen."""
        expected_dist = expected.value_counts(normalize=True)
        actual_dist = actual.value_counts(normalize=True)
        categories = sorted(set(expected_dist.index).union(set(actual_dist.index)))

        psi = 0.0
        for category in categories:
            expected_pct = expected_dist.get(category, epsilon)
            actual_pct = actual_dist.get(category, epsilon)
            psi += (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)
        return float(psi)

    @staticmethod
    def interpret_psi(psi: float) -> str:
        """Interpreteer PSI-score."""
        if psi < 0.1:
            return "geen significante drift"
        if psi < 0.25:
            return "matige drift: monitoren"
        return "sterke drift: hertraining onderzoeken"


monitor = DataDriftMonitor()

reference_categories = y_train
current_predictions = pd.Series(best_model.predict(X_test))
psi_score = monitor.population_stability_index(reference_categories, current_predictions)

print("PSI:", round(psi_score, 4))
print("Interpretatie:", monitor.interpret_psi(psi_score))

In [ ]:
def create_monitoring_snapshot(df: pd.DataFrame, predictions: pd.Series) -> Dict[str, object]:
    """Maak een monitoring snapshot voor dashboard/logging."""
    snapshot = {
        "timestamp": pd.Timestamp.utcnow().isoformat(),
        "ticket_count": int(len(df)),
        "missing_customer_message_pct": float(df["customer_message"].isna().mean() * 100),
        "avg_text_length": float(df["customer_message"].str.len().mean()),
        "source_distribution": df["source"].value_counts(normalize=True).round(4).to_dict(),
        "priority_distribution": df["priority"].value_counts(normalize=True).round(4).to_dict(),
        "prediction_distribution": predictions.value_counts(normalize=True).round(4).to_dict(),
        "psi_category_prediction": round(psi_score, 4),
    }
    return snapshot


monitoring_snapshot = create_monitoring_snapshot(
    all_tickets.sample(min(1_000, len(all_tickets)), random_state=config.random_state),
    current_predictions,
)

monitoring_path = config.output_dir / "monitoring_snapshot.json"
with open(monitoring_path, "w", encoding="utf-8") as file:
    json.dump(monitoring_snapshot, file, indent=2)

print(json.dumps(monitoring_snapshot, indent=2)[:2000])

## 17. Continuous Training strategie

Het model wordt opnieuw getraind wanneer een van deze situaties optreedt:

| Trigger | Actie |
|---|---|
| Macro F1 daalt onder 0.75 | Hertraining starten |
| PSI > 0.25 | Driftanalyse en hertraining onderzoeken |
| Nieuwe categorieën of productmodules | Labelschema bijwerken |
| Veel agent-correcties op AI-antwoorden | Kennisbank en trainingsdata actualiseren |
| Nieuwe supportkanalen | Datapipeline uitbreiden |
| Maandelijkse releasecyclus | Geplande retraining en validatie |

### Releasebeleid

1. Train nieuw model in aparte omgeving.
2. Vergelijk met productiemodel op validatie- en testset.
3. Controleer latency, foutanalyse en bias per categorie/kanaal.
4. Registreer modelversie.
5. Deploy eerst als shadow deployment of canary release.
6. Rollback bij regressie.

## 18. Governance, privacy en security

### 18.1 Privacymaatregelen

- Persoonsgegevens zoals e-mails, telefoonnummers, URLs en mentions worden geanonimiseerd.
- Alleen noodzakelijke velden worden opgeslagen.
- Ruwe klantdata wordt niet onnodig gelogd.
- Retentiebeleid moet worden afgestemd met legal/compliance.

### 18.2 Human-in-the-loop

Het cloudmodel geeft alleen een conceptantwoord. Een supportagent controleert en past het antwoord aan voordat het naar de klant wordt verzonden.

### 18.3 Bias en fairness

Mogelijke risico's:

- slechtere prestaties voor bepaalde talen;
- overclassificatie van boze berichten als hoge prioriteit;
- minder goede antwoorden voor zeldzame categorieën;
- historische bias uit oude supportantwoorden.

Mitigatie:

- prestaties meten per taal, kanaal en categorie;
- agentfeedback verzamelen;
- periodieke foutanalyse;
- duidelijke escalatie bij lage modelzekerheid.

## 19. Conclusie

In dit notebook is een compleet prototype gebouwd voor FlowSure:

- data uit meerdere bronnen wordt ingeladen en genormaliseerd;
- persoonsgegevens worden geanonimiseerd;
- een edge model classificeert supporttickets;
- prioriteit en routing worden automatisch bepaald;
- een eenvoudige retrievalcomponent maakt conceptantwoorden;
- modelartefacten, metrics en monitoringdata worden opgeslagen;
- deployment met FastAPI, Docker en CI/CD is uitgewerkt;
- monitoring en retrainingstrategie zijn beschreven.

Het prototype voldoet aan de kern van de MLOps-opdracht. Voor productie zijn vooral echte integraties, menselijke evaluatie, security hardening en uitgebreidere cloudinfrastructuur nodig.

## 20. Bronnenlijst en GAI-verantwoording

### Datasetbronnen

- Bitext. (z.d.). *Bitext Gen AI Chatbot Customer Support Dataset*. Gebruikt voor intentclassificatie en voorbeeldantwoorden.
- ThoughtVector. (z.d.). *Customer Support on Twitter Dataset*. Gebruikt voor realistische supportberichten via social media.

### Gebruikte libraries

- Pedregosa, F., et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research.
- McKinney, W. (2010). *Data Structures for Statistical Computing in Python*. Proceedings of the 9th Python in Science Conference.
- Hunter, J. D. (2007). *Matplotlib: A 2D graphics environment*. Computing in Science & Engineering.

### Generative AI-verantwoording

- ChatGPT, GPT-5.5 Thinking, 28 mei 2026. Prompt 1: *Compleet MLOps notebook voor AI-powered ticketsysteem FlowSure*. Gebruikt voor het structureren, formuleren en genereren van notebooktekst en voorbeeldcode. De inhoud is aangepast aan de projectcasus en gecontroleerd op toepasbaarheid.

> Let op: vul in je definitieve schoolversie ook de gedeelde link naar je ChatGPT-gesprek in volgens de regels van de opdracht.